# Sensitivity & Specificity vs Proteome Size

**Input files:**
- **Proteomics data (2 datasets)**: each rank-ordered descending by L2FC (column 1: protein name, column 2: L2FC)
- **TP list**: reference membrane proteins (shared across both datasets)
- **FP list**: reference nuclear proteins (shared across both datasets)

**Logic:**  
For each dataset, starting from all proteins with L2FC > 0.585, step down one protein at a time.  
At each proteome size N (top N proteins):
- **Sensitivity** = (# TP-list proteins in top N) / (total TP-list proteins) × 100
- **Specificity** = (# FP-list proteins NOT in top N) / (total FP-list proteins) × 100

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## File Paths

In [ ]:
# --- Two proteomics datasets (same TP/FP reference lists) ---
DATASETS = [
    {
        "path":  "~/Full_candidates_caax.xlsx",
        "label": "CAAX",
        "color": "steelblue",
    },
    {
        "path":  "~/Full_candidates_nrxn.xlsx",
        "label": "Nrxn",
        "color": "darkorange",
    },
]

TP_PATH    = "~/Membrane_TPlist_LCK.xlsx"
FP_PATH    = "~/Membrane_FPlist.xlsx"

L2FC_CUTOFF = 0.58

OUT_DIR = Path("~/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir : {OUT_DIR}")

## Load Data

In [ ]:
# Shared TP / FP reference lists
tp_df = pd.read_excel(TP_PATH, header=0, usecols=[0])
tp_df.columns = ["protein"]
tp_df = tp_df.dropna()

fp_df = pd.read_excel(FP_PATH, header=0, usecols=[0])
fp_df.columns = ["protein"]
fp_df = fp_df.dropna()

tp_set = set(tp_df.protein)
fp_set = set(fp_df.protein)

print(f"TP reference list: {len(tp_set)} proteins")
print(f"FP reference list: {len(fp_set)} proteins")

## Compute Sensitivity & Specificity at Each Proteome Size

In [ ]:
def compute_results(proteome_path):
    df = pd.read_excel(proteome_path, header=0, usecols=[0, 1])
    df.columns = ["protein", "L2FC"]
    df = df.dropna(subset=["L2FC"])
    df = df[df.L2FC >= L2FC_CUTOFF].sort_values("L2FC", ascending=False).reset_index(drop=True)

    sizes, sensitivity, specificity = [], [], []
    for n in range(1, len(df) + 1):
        current = set(df.protein.iloc[:n])
        sizes.append(n)
        sensitivity.append(len(current & tp_set) / len(tp_set) * 100 if tp_set else 0)
        specificity.append(len(fp_set - current) / len(fp_set) * 100 if fp_set else 0)

    print(f"{Path(proteome_path).stem}: {len(df)} proteins with L2FC >= {L2FC_CUTOFF}")
    return pd.DataFrame({"proteome_size": sizes, "sensitivity_pct": sensitivity, "specificity_pct": specificity})

for ds in DATASETS:
    ds["results"] = compute_results(ds["path"])

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ds in DATASETS:
    r = ds["results"]
    axes[0].plot(r.proteome_size, r.sensitivity_pct, color=ds["color"], lw=2, label=ds["label"])
    axes[1].plot(r.proteome_size, r.specificity_pct, color=ds["color"], lw=2, label=ds["label"])

axes[0].set_xlabel("Proteome Size", fontsize=12)
axes[0].set_ylabel("Sensitivity (%)", fontsize=12)
axes[0].set_title("Sensitivity vs Proteome Size", fontsize=13)
axes[0].set_ylim(-2, 102)
axes[0].legend(fontsize=10)

axes[1].set_xlabel("Proteome Size", fontsize=12)
axes[1].set_ylabel("Specificity (%)", fontsize=12)
axes[1].set_title("Specificity vs Proteome Size", fontsize=13)
axes[1].set_ylim(-2, 102)
axes[1].legend(fontsize=10)

plt.tight_layout()
labels = "_vs_".join(ds["label"] for ds in DATASETS)
out_path = OUT_DIR / f"{labels}_sensitivity_specificity_vs_proteome_size.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for ds in DATASETS:
    r = ds["results"]
    ax.plot(r.sensitivity_pct, r.specificity_pct, color=ds["color"], lw=2, label=ds["label"])

ax.set_xlabel("Sensitivity (%)", fontsize=12)
ax.set_ylabel("Specificity (%)", fontsize=12)
ax.set_title("Specificity vs Sensitivity", fontsize=13)
ax.set_xlim(-2, 102)
ax.set_ylim(-2, 102)
ax.legend(fontsize=10)

plt.tight_layout()
labels = "_vs_".join(ds["label"] for ds in DATASETS)
out_path = OUT_DIR / f"{labels}_specificity_vs_sensitivity.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## Proteome Size & Sensitivity at 90% Specificity

For each dataset, find the proteome size that yields a target **specificity of 90%**.
Specificity decreases as the proteome size grows (more proteins included → fewer FP-list
proteins remain excluded), so we take the **largest proteome size whose specificity is still ≥ 90%**.

At that proteome size we also read off the corresponding **sensitivity**.
These points are marked on both the specificity- and sensitivity-vs-proteome-size plots with
dashed horizontal/vertical connecting lines.

In [ ]:
TARGET_SPEC = 90.0  # target specificity (%)

for ds in DATASETS:
    r = ds["results"]
    qualifying = r[r.specificity_pct >= TARGET_SPEC]
    if len(qualifying):
        # largest proteome size whose specificity is still >= target
        row = r.loc[qualifying.proteome_size.idxmax()]
    else:
        # fallback: closest specificity to target
        row = r.loc[(r.specificity_pct - TARGET_SPEC).abs().idxmin()]

    ds["target_size"] = int(row.proteome_size)
    ds["target_spec"] = float(row.specificity_pct)
    ds["target_sens"] = float(row.sensitivity_pct)

    print(f"{ds['label']:>6}: proteome size = {ds['target_size']:>4d}  "
          f"(specificity = {ds['target_spec']:.1f}%, sensitivity = {ds['target_sens']:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ds in DATASETS:
    r = ds["results"]
    axes[0].plot(r.proteome_size, r.sensitivity_pct, color=ds["color"], lw=2, label=ds["label"])
    axes[1].plot(r.proteome_size, r.specificity_pct, color=ds["color"], lw=2, label=ds["label"])

# --- dashed connecting lines at the target (90% specificity) point ---
for ds in DATASETS:
    n   = ds["target_size"]
    spec = ds["target_spec"]
    sens = ds["target_sens"]
    c   = ds["color"]

    # Specificity panel: connect (n, spec) to both axes
    axes[1].plot([n, n], [-2, spec], color=c, ls="--", lw=1.2)
    axes[1].plot([0, n], [spec, spec], color=c, ls="--", lw=1.2)
    axes[1].plot(n, spec, "o", color=c, ms=6)
    axes[1].annotate(f"{ds['label']}: N={n}\nspec={spec:.0f}%",
                     xy=(n, spec), xytext=(8, -28), textcoords="offset points",
                     fontsize=9, color=c)

    # Sensitivity panel: at the same proteome size, mark the sensitivity
    axes[0].plot([n, n], [-2, sens], color=c, ls="--", lw=1.2)
    axes[0].plot([0, n], [sens, sens], color=c, ls="--", lw=1.2)
    axes[0].plot(n, sens, "o", color=c, ms=6)
    axes[0].annotate(f"{ds['label']}: N={n}\nsens={sens:.0f}%",
                     xy=(n, sens), xytext=(8, 8), textcoords="offset points",
                     fontsize=9, color=c)

axes[0].set_xlabel("Proteome Size", fontsize=12)
axes[0].set_ylabel("Sensitivity (%)", fontsize=12)
axes[0].set_title("Sensitivity vs Proteome Size", fontsize=13)
axes[0].set_ylim(-2, 102)
axes[0].legend(fontsize=10)

axes[1].set_xlabel("Proteome Size", fontsize=12)
axes[1].set_ylabel("Specificity (%)", fontsize=12)
axes[1].set_title("Specificity vs Proteome Size", fontsize=13)
axes[1].set_ylim(-2, 102)
axes[1].legend(fontsize=10)

plt.tight_layout()
labels = "_vs_".join(ds["label"] for ds in DATASETS)
out_path = OUT_DIR / f"{labels}_sensitivity_specificity_vs_proteome_size_at90spec.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")